# Model B
---

Load the preprocessed datasets, vocabulary, config and dataloaders using the following cell.

In [30]:
from utils import load_sets, load_vocab, load_config, get_dataloaders

train_set, val_set, test_set = load_sets()
vocab = load_vocab()
vocab_reversed = load_vocab(rev=True)
config = load_config()

# Test: filter train and valildation input sequences for questions
train_set = train_set[train_set['input'].str.strip().str.endswith('?')].reset_index(drop=True)
val_set = val_set[val_set['input'].str.strip().str.endswith('?')].reset_index(drop=True)

train_loader, val_loader, test_loader = get_dataloaders(
    train_set, val_set, test_set,
    vocab,
    config['MAX_LENGTH'],
    config['BATCH_SIZE']
)

In [31]:
train_set.head()

,input,response
0,<S0> which tool would you recommend for html a...,<S1> vi(m) <g>
1,<S0> heh <SEP> sober?,<S1> its a joke based on alcohoics anonymous
2,<S0> im trying to apt-get libopenssl-ruby and ...,"<S1> yep, throw an 'install' in there"
3,<S0> hi <SEP> when will ubuntu 11.04 be avaiab...,<S1> some time on the 28th.
4,<S0> dimming settings in power manager?,<S1> tried that and it does not work


In [32]:
config

{'MAX_LENGTH': 22,
 'MIN_TOKEN_FREQ': 5,
 'MIN_LENGTH': 2,
 'BATCH_SIZE': 64,
 'VOCAB_SIZE': 25021,
 'PAD_IDX': 0,
 'SOS_IDX': 1,
 'EOS_IDX': 2,
 'UNK_IDX': 3,
 'SEP_IDX': 4,
 'S0_IDX': 5,
 'S1_IDX': 6}

In [34]:
import torch

# Always use cuda if available (NVIDIA GPU), then try for apple silicon, otherwise just a plain old CPU (but training will be vastly slower)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'Using device: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print('Using device: MPS (Apple Silicon)')
else:
    device = torch.device("cpu")
    print('Using device: CPU')

Using device: NVIDIA GeForce RTX 4060 Laptop GPU


Quick plan:

1. Define the Encoder  — GRU that reads input, returns all hidden states + final hidden state (With attension)
2. Define the Decoder  — GRU that generates output token by token by looking back at all encoder states and compute a weighted average 
                         of the most relevant parts of the input sentence before predicting the next word. (With attension)
4. Define the Seq2Seq  — wrapper that connects encoder and decoder, handles teacher forcing
5. Training loop       — with validation loss monitoring
6. Inference function  — greedy decoding for generating responses
7. Evaluation          — BLEU score on test set + manual examples

## Encoder

---

The encoder will read the input sequences we've prepared, and will output a single final hidden state to initialise the decoder. So, the architecture is actually quite simple:
- An embedding layer (dataset has low GloVe coverage, so we can train embeddings from scratch).
- A stack of Gated Recurrent Units (combats vanishing gradients, fewer parameters to train than an LSTM, with comparable performance).
- Dropout as a regularisation technique to combat overfitting and improve generalisation (only applied during training). We'll apply dropout between the embedding layer, and between the layers in the GRU.

Defining a sequential model in PyTorch follows the usual pattern:
- Extend the [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) base class.
- Initialise your layers in the constructor.
- Override the `forward` method, explicitly feeding X inputs through your layers.

Please read the [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html) documentation, it explains a lot! :)

In [40]:
from encoder import Encoder

## Decoder

---

The decoder must first initialise its hidden state with the final hidden state of the encoder. Then, at each timestep (a forward pass through the network processing one token):

1- Obtain the embedding for the previous token

2- Pass it through the GRU

3- Compute attention scores between the decoder output and the encoder outputs

4- Use the attention weights to compute a context vector as a weighted sum of encoder hidden states

5- Combine the decoder output and the context vector

6- Project the combined representation to the vocabulary size to predict the next token (using a fully connected linear layer)

Outside of this, its layer initialisation is very similar to the encoder.

Note that we also train separate embeddings in the decoder rather than using shared embeddings between the encoder and decoder (called weight tying). This keeps the embeddings specialised, as the encoder and decoder objectives differ: the encoder must encode the meaning of the input sequence into hidden representations, while the decoder must use these representations to predict the next token given the previous one. Allowing each component to learn token representations optimised for its objective improves modelling flexibility.

In [43]:
from decoder import Decoder

## Seq2Seq Wrapper

---

This is the wrapper model that will be built from the encoder and decoder. It will:
1. Pass the full input sequence through the encoder to produce a final hidden state
2. Use this final hidden state to initialise the decoder
3. Loop through each decoding timestep and feed each token in one at a time
4. Apply **teacher forcing** during training only (feeding the ground truth labels in at each timestep to improve stability and reduce _exposure bias_I)

The variables passed into the `forward` method are those we prepared at the data preparation stage:
- `encoder_input`: the encoded input sequence (for the encoder)
- `decoder_input`: the encoded response with SOS prepended

In [46]:
from seq2Seq import Seq2Seq

## Training Loop

---

This is the main training loop, where we'll use cross entropy loss, Adam optimiser, and gradient clipping to manage exploding gradients (this is common in BPTT).

We can use a `max_norm=1.0` to scale the gradient vector down if its norm exceeds 1 to help keep the weight updates stable.

First things first: let's define the training and model hyperparameters, and instantiate the model.

In [49]:
# Training Parameters
EPOCHS = 10
CLIP_MAX_NORM = 40.0
TEACHER_FORCING_PROBA = 0.5

# Model Hyperparameters
ENCODER_EMBED_DIM = 128
ENCODER_HIDDEN_DIM = 256
ENCODER_NUM_LAYERS = 2
ENCODER_DROPOUT_PROBA = 0.2

DECODER_EMBED_DIM = 128
DECODER_HIDDEN_DIM = 256
DECODER_NUM_LAYERS = 2
DECODER_DROPOUT_PROBA = 0.2

LEARNING_RATE=1e-4

model_b = Seq2Seq(
    encoder=Encoder(
        vocab_size=config['VOCAB_SIZE'], 
        embed_dim=ENCODER_EMBED_DIM, 
        padding_idx=config['PAD_IDX'], 
        hidden_dim=ENCODER_HIDDEN_DIM,
        num_layers=ENCODER_NUM_LAYERS,
        dropout_proba=ENCODER_DROPOUT_PROBA,
        use_attention=True
    ),
    decoder = Decoder(
        vocab_size=config['VOCAB_SIZE'],
        embed_dim=DECODER_EMBED_DIM,
        padding_idx=config['PAD_IDX'],
        hidden_dim=DECODER_HIDDEN_DIM,
        num_layers=DECODER_NUM_LAYERS,
        dropout_proba=DECODER_DROPOUT_PROBA,
        use_attention=True
    ),
    device=device,
    use_attention=True
).to(device) # Move model to device

In [51]:
#!mkdir -p models/model_b
from pathlib import Path

Path("models/model_b").mkdir(parents=True, exist_ok=True)

In [53]:
from torch.optim import Adam
from pathlib import Path
import time
import torch
import torch.nn as nn
from bleu import compute_bleu

checkpoint_dir = Path('models/model_b')
model_b_baseline_state_dict = checkpoint_dir / 'model_b_baseline.pt'

criterion = nn.CrossEntropyLoss(ignore_index=config['PAD_IDX'])
optimiser = Adam(model_b.parameters(), lr=LEARNING_RATE)

# Latest checkpoint to resume from
existing_checkpoints = sorted(checkpoint_dir.glob('checkpoint_epoch_*.pt'))

if model_b_baseline_state_dict.exists():
    print('Loading baseline model weights...')
    model_b.load_state_dict(torch.load(model_b_baseline_state_dict, weights_only=True))
    start_epoch = EPOCHS
elif existing_checkpoints:
    latest = existing_checkpoints[-1]
    checkpoint = torch.load(latest, weights_only=True)
    model_b.load_state_dict(checkpoint['model_state_dict'])
    optimiser.load_state_dict(checkpoint['optimiser_state_dict'])
    start_epoch = len(existing_checkpoints)
    print(f'Resuming from epoch {start_epoch}/{EPOCHS}')
else:
    print('No checkpoints, training model from scratch...')
    start_epoch = 0
    
def train_epoch(model, loader, optimiser, criterion, device, clip_max_norm, teacher_forcing_proba):
    model.train() # Enables training mode (includes enabling dropout)
    total_loss = 0
    total_batches = len(loader)
    log_interval = max(1, int(total_batches * 0.05)) # Every 5%

    for batch_idx, (encoder_input_batch, decoder_input_batch, decoder_target_batch) in enumerate(loader):
        # Remember tensors and models must be on the same device
        encoder_input_batch = encoder_input_batch.to(device)
        decoder_input_batch = decoder_input_batch.to(device)
        decoder_target_batch = decoder_target_batch.to(device)

        optimiser.zero_grad()
        predictions = model(encoder_input_batch, decoder_input_batch, teacher_forcing_proba=teacher_forcing_proba, padding_idx=config['PAD_IDX']) # Forward pass
        predictions = predictions.permute(0, 2, 1) # Need to reshape for CrossEntropyLoss, this just reorders dimensions
        loss = criterion(predictions, decoder_target_batch) # Compute loss
        loss.backward() # BPTT
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_max_norm) # Gradient clipping comes before weight updates
        optimiser.step() # Update weights
        total_loss += loss.item() 

        # Log every 25% of batches (or whatever log_percent you set)
        if (batch_idx + 1) % log_interval == 0:
            avg_loss = total_loss / (batch_idx + 1)
            progress = 100 * (batch_idx + 1) / total_batches
            print(f'  [{progress:.0f}%] Batch {batch_idx+1}/{total_batches} | Avg Loss: {avg_loss:.4f}')

    return total_loss / len(loader) # Average loss per batch

def val_epoch(model, loader, criterion, device):
    model.eval() # Deactivates dropout
    total_loss = 0
    with torch.no_grad(): # Disable gradient computation and tracking
        for encoder_input_batch, decoder_input_batch, decoder_target_batch in loader:
            encoder_input_batch = encoder_input_batch.to(device)
            decoder_input_batch = decoder_input_batch.to(device)
            decoder_target_batch = decoder_target_batch.to(device)

            # During evaluation there should be no teacher forcing
            predictions = model(encoder_input_batch, decoder_input_batch, teacher_forcing_proba=0.0, padding_idx=config['PAD_IDX'])
            predictions = predictions.permute(0, 2, 1)
            loss = criterion(predictions, decoder_target_batch)
            total_loss += loss.item()

    return total_loss / len(loader)

# Main training loop
epoch_times = []
for epoch in range(start_epoch, EPOCHS):
    start = time.time()
    
    train_loss = train_epoch(
        model_b, 
        train_loader, 
        optimiser, 
        criterion, 
        device, 
        CLIP_MAX_NORM,
        TEACHER_FORCING_PROBA
    )
    val_loss = val_epoch(
        model_b, 
        val_loader, 
        criterion, 
        device
    )
    bleu = compute_bleu(
        model_b, 
        val_loader,
        vocab_reversed, 
        config, 
        device
    )
    elapsed = time.time() - start
    epoch_times.append(elapsed)
    
    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val BLEU: {bleu:.4f} | Time: {elapsed:.2f}s')

    # Checkpoint
    torch.save({
        'model_state_dict': model_b.state_dict(), # The actual model and its weights
        'optimiser_state_dict': optimiser.state_dict(), #
    }, checkpoint_dir / f'checkpoint_epoch_{epoch+1}.pt')

# Save model's state_dict as per https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html#saving-loading-model-for-inference
torch.save(model_b.state_dict(), model_b_baseline_state_dict)

# Print benchmarks
if epoch_times:
    print(f'\n--- Benchmark Summary ---')
    print(f'Device: {device}')
    print(f'Total training time: {sum(epoch_times):.2f}s')
    print(f'Average epoch time: {sum(epoch_times)/len(epoch_times):.2f}s')
    print(f'Fastest epoch: {min(epoch_times):.2f}s')
    print(f'Slowest epoch: {max(epoch_times):.2f}s')
else:
    print('Training already complete!')

No checkpoints, training model from scratch...
  [5%] Batch 67/1359 | Avg Loss: 8.8449
  [10%] Batch 134/1359 | Avg Loss: 7.8575
  [15%] Batch 201/1359 | Avg Loss: 7.4925
  [20%] Batch 268/1359 | Avg Loss: 7.3012
  [25%] Batch 335/1359 | Avg Loss: 7.1807
  [30%] Batch 402/1359 | Avg Loss: 7.0995
  [35%] Batch 469/1359 | Avg Loss: 7.0412
  [39%] Batch 536/1359 | Avg Loss: 6.9955
  [44%] Batch 603/1359 | Avg Loss: 6.9586
  [49%] Batch 670/1359 | Avg Loss: 6.9290
  [54%] Batch 737/1359 | Avg Loss: 6.9022
  [59%] Batch 804/1359 | Avg Loss: 6.8813
  [64%] Batch 871/1359 | Avg Loss: 6.8609
  [69%] Batch 938/1359 | Avg Loss: 6.8429
  [74%] Batch 1005/1359 | Avg Loss: 6.8275
  [79%] Batch 1072/1359 | Avg Loss: 6.8144
  [84%] Batch 1139/1359 | Avg Loss: 6.8024
  [89%] Batch 1206/1359 | Avg Loss: 6.7918
  [94%] Batch 1273/1359 | Avg Loss: 6.7818
  [99%] Batch 1340/1359 | Avg Loss: 6.7721
Computing BLEU for epoch...
  BLEU: batch 1/10...
  BLEU: batch 4/10...
  BLEU: batch 7/10...
  BLEU: batch 1

In [55]:
from utils import encode_sequence, clean

def generate_response(model, text, vocab, vocab_reversed, config, device, max_len=30, temperature=1.0, top_k=5):
    model.eval()
    encoder_input = torch.tensor(
        encode_sequence(clean(text), vocab, config['MAX_LENGTH']),
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        output, hidden = model.encoder(encoder_input)
        next_token = torch.tensor([config['SOS_IDX']], dtype=torch.long).to(device)
        tokens = []

        for _ in range(max_len):
            pred, hidden = model.decoder(next_token, hidden,output)
            scaled = pred / temperature
            top_k_values, top_k_indices = torch.topk(scaled, top_k, dim=-1)
            probs = torch.softmax(top_k_values, dim=-1)
            next_token = top_k_indices.gather(-1, torch.multinomial(probs, 1)).squeeze(-1)
            if next_token.item() == config['EOS_IDX']:
                break
            tokens.append(vocab_reversed[str(next_token.item())])

    return ' '.join(tokens)

In [59]:
# Inference
vocab_reversed = load_vocab(rev=True)

test_inputs = [
    'How do I install python3?',
    'Hello world',
    'Not working'
]

for text in test_inputs:
    response = generate_response(model_b, text, vocab, vocab_reversed, config, device)
    print(f'Input:    {text}')
    print(f'Response: {response}\n')

Input:    How do I install python3?
Response: <UNK> <UNK> the

Input:    Hello world
Response: i

Input:    Not working
Response: <UNK>

